# Project 01: Exploratory Data Analysis (EDA) on Titanic

**Goal:** load the Titanic passenger data, clean it, and answer questions about who was most likely to survive, using pandas.

## My three questions
I wrote these before touching the data so the analysis stays focused.

1. **Did women survive more than men?**
2. **Did ticket class matter?**
3. **Was age related to survival?**

## 1. Load and look

In [1]:
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

df = sns.load_dataset("titanic")
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 100.4 KB


In [3]:
df.describe().round(2)

,survived,pclass,age,sibsp,parch,fare
count,891.00,891.00,714.00,891.00,891.00,891.00
mean,0.38,2.31,29.70,0.52,0.38,32.20
std,0.49,0.84,14.53,1.10,0.81,49.69
min,0.00,1.00,0.42,0.00,0.00,0.00
25%,0.00,2.00,20.12,0.00,0.00,7.91
50%,0.00,3.00,28.00,0.00,0.00,14.45
75%,1.00,3.00,38.00,1.00,0.00,31.00
max,1.00,3.00,80.00,8.00,6.00,512.33


### What each column means
| Column | Meaning |
|---|---|
| `survived` | 1 = survived, 0 = did not survive (**the target**) |
| `pclass` / `class` | Ticket class (1 = first, 3 = third) |
| `sex` | Male or female |
| `age` | Age in years |
| `sibsp` | Number of siblings or spouses aboard |
| `parch` | Number of parents or children aboard |
| `fare` | Ticket price paid |
| `embarked` / `embark_town` | Port where the passenger boarded (C, Q, S) |
| `who` | man, woman or child |
| `adult_male` | True if an adult man |
| `deck` | Cabin deck letter |
| `alive` | "yes" / "no" copy of `survived` |
| `alone` | True if travelling without family |

## 2. Find missing values

In [4]:
missing = df.isna().sum()
pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(1)}).query("missing > 0")

,missing,percent
age,177,19.9
embarked,2,0.2
deck,688,77.2
embark_town,2,0.2


`age` is missing for about 20% of passengers, `deck` for about 77%, and `embarked` / `embark_town` for 2 passengers each.

## 3. Fix them, with a reason for each choice
| Column | Fix | Why |
|---|---|---|
| `age` | Fill with the **median age of the same class and sex** | Age differs a lot between groups (first-class women were older than third-class men), so a group median is more realistic than one overall value. The median is not pulled around by a few very old or very young passengers. |
| `deck` | **Drop the column** | About 77% is empty, so any fill would be mostly invented. |
| `embarked`, `embark_town` | Fill with the **most common value** | Only 2 rows are missing, so the effect on results is tiny. |

I am **not** dropping rows, because that would throw away real passengers and could bias the survival rate.

In [5]:
df["age"] = df["age"].fillna(df.groupby(["pclass", "sex"])["age"].transform("median"))
df = df.drop(columns=["deck"])
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

print("Missing values left:", int(df.isna().sum().sum()))

Missing values left: 0


## 4. Check duplicates and data types

In [6]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 118


The count above is not zero, but I am **keeping those rows**. This dataset has no passenger ID column, so two different passengers with the same class, age, fare and family size look identical. They are most likely real, different people rather than copy-paste errors. Dropping them would remove real passengers.

In [7]:
for col in ["sex", "embarked", "who", "embark_town", "alive"]:
    df[col] = df[col].astype("category")   # text labels that repeat are categories

df.dtypes

survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
embarked       category
class          category
who            category
adult_male         bool
embark_town    category
alive          category
alone              bool
dtype: object

## 5. Group and compare
I compare survival rate by **sex**, **ticket class** and **age band** (three factors), plus two extra factors, travelling alone and boarding port.

In [8]:
def rate_table(col):
    t = df.groupby(col, observed=True)["survived"].agg(survival_rate="mean", passengers="count")
    t["survival_rate"] = (t["survival_rate"] * 100).round(1)
    return t.sort_values("survival_rate", ascending=False)

df["age_band"] = pd.cut(df["age"], [0, 12, 18, 40, 60, 100],
                        labels=["Child", "Teen", "Adult", "Middle-aged", "Senior"])

print("Overall survival rate: %.1f%%" % (df["survived"].mean() * 100))
for col in ["sex", "pclass", "age_band"]:
    print(f"\nSurvival rate (%) by {col}")
    display(rate_table(col))

Overall survival rate: 38.4%

Survival rate (%) by sex


,survival_rate,passengers
sex,,
female,74.2,314
male,18.9,577



Survival rate (%) by pclass


,survival_rate,passengers
pclass,,
1,63.0,216
2,47.3,184
3,24.2,491



Survival rate (%) by age_band


,survival_rate,passengers
age_band,,
Child,58.0,69
Teen,42.9,70
Middle-aged,39.1,128
Adult,36.0,602
Senior,22.7,22


In [9]:
for col in ["alone", "embark_town"]:
    print(f"\nSurvival rate (%) by {col}")
    display(rate_table(col))


Survival rate (%) by alone


,survival_rate,passengers
alone,,
False,50.6,354
True,30.4,537



Survival rate (%) by embark_town


,survival_rate,passengers
embark_town,,
Cherbourg,55.4,168
Queenstown,39.0,77
Southampton,33.9,646


In [10]:
print("Survival rate (%) by sex and class")
(df.pivot_table(values="survived", index="sex", columns="pclass", observed=True) * 100).round(1)

Survival rate (%) by sex and class


pclass,1,2,3
sex,,,
female,96.8,92.1,50.0
male,36.9,15.7,13.5


## 6. Answer my questions

In [11]:
sex_rate  = df.groupby("sex", observed=True)["survived"].mean() * 100
cls_rate  = df.groupby("pclass")["survived"].mean() * 100
band_rate = df.groupby("age_band", observed=True)["survived"].mean() * 100
band_n    = df.groupby("age_band", observed=True)["survived"].count()

display(Markdown(f"""
**Q1. Did women survive more than men?**
Yes. Women survived at about **{sex_rate['female']:.0f}%** compared with about **{sex_rate['male']:.0f}%** for men.

**Q2. Did ticket class matter?**
Yes. First class survived at about **{cls_rate[1]:.0f}%**, second class at **{cls_rate[2]:.0f}%** and third class at **{cls_rate[3]:.0f}%**.

**Q3. Was age related to survival?**
Yes, but less strongly than sex or class. Children (12 and under) survived at about **{band_rate['Child']:.0f}%**, the highest of any age band, while adults aged 19 to 40 survived at about **{band_rate['Adult']:.0f}%**. Seniors survived at about **{band_rate['Senior']:.0f}%**, but there are only {band_n['Senior']} of them, so that figure is noisy.
"""))


**Q1. Did women survive more than men?**
Yes. Women survived at about **74%** compared with about **19%** for men.

**Q2. Did ticket class matter?**
Yes. First class survived at about **63%**, second class at **47%** and third class at **24%**.

**Q3. Was age related to survival?**
Yes, but less strongly than sex or class. Children (12 and under) survived at about **58%**, the highest of any age band, while adults aged 19 to 40 survived at about **36%**. Seniors survived at about **23%**, but there are only 22 of them, so that figure is noisy.


## 7. Summary of my three findings

In [12]:
overall = df["survived"].mean() * 100
display(Markdown(f"""
1. **Sex was the biggest factor.** {sex_rate['female']:.0f}% of women survived versus {sex_rate['male']:.0f}% of men, out of an overall rate of {overall:.0f}%.
2. **Ticket class mattered a lot.** Survival fell from {cls_rate[1]:.0f}% in first class to {cls_rate[3]:.0f}% in third class.
3. **Children had the best odds by age.** About {band_rate['Child']:.0f}% of children survived, versus {band_rate['Adult']:.0f}% of adults aged 19 to 40.

**Careful wording:** these are *associations*, not proof of cause. Sex, class and fare are linked to each other (for example, more women were in first class), so I cannot say any one factor alone caused survival.
"""))


1. **Sex was the biggest factor.** 74% of women survived versus 19% of men, out of an overall rate of 38%.
2. **Ticket class mattered a lot.** Survival fell from 63% in first class to 24% in third class.
3. **Children had the best odds by age.** About 58% of children survived, versus 36% of adults aged 19 to 40.

**Careful wording:** these are *associations*, not proof of cause. Sex, class and fare are linked to each other (for example, more women were in first class), so I cannot say any one factor alone caused survival.
